# 08C — Learn and test interpretable condition–outcome rules

**Question:** Can a small number of environmental and fireline-type conditions identify groups with different recorded Held/Burned Over outcomes on fires excluded from rule development?

Unlike the earlier surrogate tree, these trees learn **recorded NIFC outcomes directly**, not LightGBM probabilities. This notebook does not change the existing LightGBM reference, outcome labels, fireline units, or prediction times.

Save the notebook, then run all cells in Antigravity with the project **.venv** kernel. No downloads or GPU are needed. Outputs go to a new timestamped folder, with each fitted model saved separately. If interrupted, restart from the configuration cell to create a fresh run; automatic checkpoint resumption is not implemented.

## What will be learned and evaluated

1. Five-fold fire-held-out tests of a fixed shallow-tree procedure.
2. Expanding-year tests: 2020→2021 through 2020–2023→2024.
3. **Frozen rule cards:** one tree trained on 2020–2022, applied unchanged to 2023 and 2024.
4. Matched comparisons with simpler predictor sets and with/without reported width.
5. Held fractions, uncertainty, group coverage, within-fire differences, and predictive error.

The previously explored archive remains **exploratory research evidence**. Fires excluded from a particular fit are not an independently collected confirmation dataset. These are associations with recorded outcomes, not effects of constructing, widening, or choosing a type of fireline.

## Observation level and scope

The local fireline units are retained because parts of an original mapped record can have different environmental conditions and prediction times. Source-record weights prevent repeated children from automatically gaining more influence. We do **not** assign one weather value to an entire original fireline.

Archived fireline type and width have not been verified as available at historical prediction time. Models using them describe retrospective condition–outcome relationships. Source-record length is excluded from rule predictors because outcome mapping can influence that length. Original-record construction analyses remain in 08B.


## 0. Fixed configuration

The verified 08B run is pinned below. Settings and predictor sets are saved before fitting. There is no search over tree depths, calibration methods, or thresholds.

The primary condition tree has at most three splits along a path. Branches with too little **training** support are collapsed. Test outcomes never determine tree structure, preprocessing, pruning, or fitted probabilities.

By default uncertainty uses 2,000 resamples of complete fires. A small smoke-test count is only for software verification, not paper results.


In [ ]:
import os  # Configurable project root for the public release.
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version
import json, hashlib, sys, platform, uuid, zlib, time, importlib.util
required=["numpy","pandas","pyarrow","scikit-learn","matplotlib","IPython"]
for module in ["numpy","pandas","pyarrow","sklearn","matplotlib","IPython"]:
    if importlib.util.find_spec(module) is None:
        raise ImportError("Use the project .venv kernel; missing package: "+module)
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

def find_root():
    for p in [Path.cwd(),*Path.cwd().parents,Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))]:
        if (p/"data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet").is_file():
            return p.resolve()
    raise FileNotFoundError("Open the notebook in the project or set PROJECT_ROOT.")
PROJECT_ROOT=find_root()
INPUT_RUN_NAME="08B_20260930T224636_898345Z_38d03c"
INPUT_DIR=PROJECT_ROOT/"outputs/08_condition_outcomes"/INPUT_RUN_NAME
UNIT_PATH=INPUT_DIR/"08B_existing_units_with_diagnostic_weights.parquet"
PARENT_PATH=INPUT_DIR/"08B_retained_source_record_summaries.parquet"
MASTER_PATH=PROJECT_ROOT/"data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
NOTEBOOK_PATH=PROJECT_ROOT/"notebooks/08C_interpretable_condition_rules.ipynb"
RANDOM_SEED=20261001
BOOTSTRAP_REPLICATES=2000
MAX_DEPTH=3
MIN_LEAF_UNITS=500
MIN_LEAF_PARENTS=100
MIN_LEAF_FIRES=10
MIN_LEAF_WEIGHT_FRACTION=0.01
MIN_TEST_UNITS=200
MIN_TEST_PARENTS=50
MIN_TEST_FIRES=10
MIN_CONTRAST_PARENTS_PER_FIRE=5
WIDTH_REVIEW_FT=300.0
FROZEN_TRAIN_YEARS=[2020,2021,2022]
FROZEN_TEST_YEARS=[2023,2024]
RUN_WIDTH_ANALYSIS=True
IS_SMOKE_TEST=False
BUILD_ID="08C_V1_DIRECT_OUTCOME_SHALLOW_RULES"
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")+"_"+uuid.uuid4().hex[:6]
RUN_DIR=PROJECT_ROOT/"outputs/08_condition_outcomes"/("08C_"+RUN_ID)
MODEL_DIR=RUN_DIR/"models"
FIG_DIR=RUN_DIR/"figures"

# key: (source field, plain-language description, measurement unit, allowed lower/upper bounds)
PREDICTORS={
 "humidity":("hrrr_window_0_18h_rh2_pct_mean","Mean forecast relative humidity, 0–18 h","%",(0,100)),
 "gust":("hrrr_gust_mps_p18h","Forecast gust at +18 h","m/s",(0,None)),
 "wind":("hrrr_window_0_18h_wind10_speed_mean_mps","Mean forecast wind speed, 0–18 h","m/s",(0,None)),
 "temperature":("hrrr_window_0_18h_t2_c_max","Maximum forecast temperature, 0–18 h","deg C",(None,None)),
 "slope":("terrain_slope_mean_deg","Mean terrain slope","degrees",(0,90)),
 "elevation":("terrain_elevation_mean_m","Mean elevation","m",(None,None)),
 "crown_depth":("landfire_line_crown_depth_mean","Mean crown depth sampled on the fireline","m",(0,None)),
 "canopy_cover":("landfire_line_cc_mean","Mean canopy cover sampled on the fireline","%",(0,100)),
 "distance":("local_fire_to_line_distance_geom_m","Mapped fire-perimeter distance","m",(0,None)),
 "width":("LineWidthFeet","Reported fireline width","m",(0,WIDTH_REVIEW_FT*0.3048)),
}
def saved_notebook_source_hash():
    if not NOTEBOOK_PATH.is_file(): return None
    doc=json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    return hashlib.sha256(json.dumps([c["source"] for c in doc["cells"]],ensure_ascii=False).encode()).hexdigest()
NOTEBOOK_SOURCE_HASH_AT_START=saved_notebook_source_hash()
ENVIRONMENT_KEYS=[k for k in PREDICTORS if k!="width"]
FUEL_GROUPS={"gr":"Grass","gs":"Grass–shrub","sh":"Shrub","tu":"Timber understory",
             "tl":"Timber litter","sb":"Slash/blowdown","nb":"Nonburnable"}
FUEL_KEYS=[]
for group,label in FUEL_GROUPS.items():
    key="fuel_"+group
    PREDICTORS[key]=("landfire_line_fbfm_group_"+group+"_fraction",
                     label+" fuel-group fraction sampled on the fireline","fraction",(0,1))
    FUEL_KEYS.append(key)
pd.set_option("display.max_rows",30)
pd.set_option("display.max_columns",18)
plt.rcParams.update({"figure.dpi":110,"savefig.dpi":220,"axes.spines.top":False,"axes.spines.right":False})
print(BUILD_ID,"\nInput:",INPUT_DIR,"\nNew output:",RUN_DIR)


## 1. Load the verified population without changing it

The outcome is Held = 1 and Burned Over = 0. Not Engaged does not enter this binary analysis.

The notebook checks the 08B completion record, unique fireline-unit identifiers, source-record membership, saved fire folds, and prediction times. Invalid predictor values are recorded as missing for modeling; source columns remain unchanged. Width in feet is converted to metres only in a separate modeling column.

The primary vegetation predictors are **line-sampled canopy structure**. A separately specified fuel-composition sensitivity adds seven existing broad LANDFIRE fuel-group fractions. Those fractions describe the pre-event underlying landscape, not the constructed treatment, and are not measured fuel moisture. Exact fuel-model codes are never treated as ordered numbers. Opposite-side canopy is excluded from the main rules because its missingness depends on approach-direction availability.


In [ ]:
def require(ok,message):
    if not bool(ok): raise AssertionError(message)

def safe_json(x):
    if isinstance(x,dict): return {str(k):safe_json(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [safe_json(v) for v in x]
    if isinstance(x,np.ndarray): return safe_json(x.tolist())
    if isinstance(x,Path): return str(x)
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.bool_,)): return bool(x)
    if isinstance(x,(float,np.floating)):
        if not np.isfinite(x): return None
        return float(x)
    return x

def write_json(path,obj):
    Path(path).write_text(json.dumps(safe_json(obj),indent=2,allow_nan=False),encoding="utf-8")

def digest(frame,columns,keys):
    f=frame[columns].sort_values(keys).reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(f,index=False).to_numpy().tobytes()).hexdigest()

for p in [UNIT_PATH,PARENT_PATH,MASTER_PATH,INPUT_DIR/"08B_run_manifest.json",INPUT_DIR/"08B_quality_checks.json"]:
    require(p.is_file(),"Missing input: "+str(p))
prior=json.loads((INPUT_DIR/"08B_run_manifest.json").read_text(encoding="utf-8"))
quality=json.loads((INPUT_DIR/"08B_quality_checks.json").read_text(encoding="utf-8"))
require(not prior["is_smoke_test"] and not quality["is_smoke_test"],"08C requires the completed production 08B run.")
for key in ["source_records_preserved","frozen_units_and_prediction_times_preserved","all_source_joins_match","not_engaged_kept_separate"]:
    require(quality[key],"08B did not pass "+key)

units=pd.read_parquet(UNIT_PATH).sort_values("unit_uid").reset_index(drop=True)
original_columns=list(units.columns)
require(len(units)==304143 and units["_irwin_norm"].nunique()==381,"Unexpected frozen population.")
require(units["unit_uid"].notna().all() and units["unit_uid"].is_unique,"Missing/duplicate unit ID.")
require(units["source_record_key"].notna().all() and units["source_record_key"].nunique()==142999,"Unexpected source membership.")
require(units["y"].isin([0,1]).all() and units["y"].eq(units["hold_label"]).all(),"Outcome mismatch.")
require(units["prediction_anchor_utc"].notna().all() and units["prediction_anchor_utc"].equals(units["exact_t0_observation_utc"]),"Prediction-time mismatch.")
for col in ["Year","outer_fold"]:
    require(units.groupby("_irwin_norm")[col].nunique().eq(1).all(),"A fire spans multiple "+col+" values.")
for col in ["_irwin_norm","y","Year","outer_fold"]:
    require(units.groupby("source_record_key")[col].nunique().eq(1).all(),"Parent inconsistent: "+col)
require(set(units["outer_fold"])==set(range(5)),"Expected saved folds 0–4.")
input_hash=digest(units,original_columns,["unit_uid"])
reference=pd.read_parquet(MASTER_PATH,columns=["unit_uid","hold_label","prediction_anchor_utc","exact_t0_observation_utc"]).sort_values("unit_uid").reset_index(drop=True)
pd.testing.assert_frame_equal(units[reference.columns],reference,check_dtype=False)
parent_metadata=pd.read_parquet(PARENT_PATH,columns=["source_record_key","n_prediction_times","retained_length_ratio"])
require(parent_metadata["source_record_key"].is_unique,"Duplicate parent metadata.")
units=units.merge(parent_metadata,on="source_record_key",how="left",validate="many_to_one")
require(units["n_prediction_times"].notna().all(),"Parent metadata missing.")
units["row_id"]=np.arange(len(units),dtype=np.int64)
fuel_columns=[PREDICTORS[k][0] for k in FUEL_KEYS]
fuel_frame=pd.read_parquet(MASTER_PATH,columns=["unit_uid"]+fuel_columns)
require(fuel_frame["unit_uid"].is_unique,"Duplicate fuel join key.")
units=units.merge(fuel_frame,on="unit_uid",how="left",validate="one_to_one")
require(np.allclose(units[fuel_columns].sum(axis=1),1.0) and units[fuel_columns].notna().all().all(),
        "Fuel-group fractions must be complete and sum to one.")
predictor_audit=[]
for key,(source,label,unit,bounds) in PREDICTORS.items():
    x=pd.to_numeric(units[source],errors="coerce")
    if key=="width": x=x*0.3048
    lo,hi=bounds
    valid=pd.Series(np.isfinite(x),index=x.index)
    if lo is not None: valid &= x.ge(lo)
    if hi is not None: valid &= x.le(hi)
    if key=="width": valid &= x.gt(0)
    units["x_"+key]=x.where(valid)
    predictor_audit.append({"predictor":key,"source_field":source,"description":label,"unit":unit,
                            "n_missing_or_invalid":int((~valid).sum()),"n_valid":int(valid.sum())})
units["type_value"]=units["fireline_type"].fillna("__MISSING__").astype(str)
units["width_available"]=units["x_width"].notna()
require(units["width_available"].eq(units["width_status"].eq("Positive, at or below review flag")).all(),
        "Width screen differs from the verified 08B screen; revise both explicitly before changing the review flag.")
RUN_DIR.mkdir(parents=True,exist_ok=False)
MODEL_DIR.mkdir();FIG_DIR.mkdir()
input_paths=[UNIT_PATH,PARENT_PATH,MASTER_PATH,INPUT_DIR/"08B_run_manifest.json",INPUT_DIR/"08B_quality_checks.json"]
input_stats={str(p):{"bytes":p.stat().st_size,"modified_ns":p.stat().st_mtime_ns} for p in input_paths}
tables={"predictor_inventory":pd.DataFrame(predictor_audit)}
tables["population_by_year"]=units.groupby("Year").agg(
    fireline_units=("unit_uid","size"),source_records=("source_record_key","nunique"),fires=("_irwin_norm","nunique"),
    held=("y","sum"),width_available=("width_available","sum")).reset_index()
tables["population_by_fold"]=units.groupby("outer_fold").agg(
    fireline_units=("unit_uid","size"),source_records=("source_record_key","nunique"),fires=("_irwin_norm","nunique")).reset_index()
units[["row_id","unit_uid","_irwin_norm","source_record_key","Year","outer_fold","y","prediction_anchor_utc",
       "actual_line_length_m","width_available"]].to_parquet(RUN_DIR/"08C_unit_lookup.parquet",index=False)
display(tables["population_by_year"])
display(tables["predictor_inventory"])


## 2. Weighting, baselines, and uncertainty

**Training:** each fire has equal total weight; each source record within a fire has equal weight; that record's weight is divided among its retained fireline units. No outcome class is artificially balanced. A fixed sensitivity model instead gives each source record equal total weight across the archive.

**Baseline:** the constant Held probability is calculated from the training outcomes under the same training weights as the fitted tree. It is never estimated from test outcomes. The frozen tree also keeps its 2020–2022 baseline unchanged in both test years.

**Primary error:** first average squared probability errors using source-record weights within each fire, then average equally across fires. We call this *fire-average, source-balanced Brier error*. It is not identical to the manuscript's fire-average error, which gave fireline units equal weight within a fire.

For readable rule cards, source weights are allocated over the full evaluated population before examining a leaf. Within a leaf, we calculate one weighted Held fraction per represented fire and average those fire-level fractions equally. A source record split across leaves contributes only its allocated share to each leaf.

The fitted leaf probability uses the training weights allocated before splitting into leaves. The displayed leaf Held fraction instead gives every represented fire equal weight within that leaf. These summaries need not agree even on training data; comparing them is not a calibration test.

Intervals resample whole fires with predictions and rules fixed. They do not include retraining uncertainty or correct for prior exploration, selection bias, or multiple comparisons. Empty, sparse, and single-fire groups remain visible.


In [ ]:
def source_weights(frame):
    require(len(frame)>0,"Cannot weight an empty population.")
    counts=frame.groupby("source_record_key")["source_record_key"].transform("size")
    w=1.0/counts.to_numpy(dtype=float)
    totals=pd.Series(w,index=frame.index).groupby(frame["source_record_key"]).sum()
    require(np.allclose(totals,1.0),"Source weights do not sum to one.")
    return w

def training_weights(frame,mode="fire_source"):
    w=source_weights(frame)
    if mode=="fire_source":
        parents=frame[["_irwin_norm","source_record_key"]].drop_duplicates().groupby("_irwin_norm").size()
        w=w/frame["_irwin_norm"].map(parents).to_numpy(dtype=float)
        total=pd.Series(w,index=frame.index).groupby(frame["_irwin_norm"]).sum()
        require(np.allclose(total,1.0),"Fire-balanced training weights do not sum to one per fire.")
    else:
        require(mode=="source","Unknown training weighting.")
    return w

def bootstrap_interval(values,fire_ids):
    x=np.asarray(values,dtype=float)
    require(len(x)==len(fire_ids) and np.isfinite(x).all(),"Invalid fire values for bootstrap.")
    if len(x)<2: return np.nan,np.nan
    order=np.argsort(np.asarray(fire_ids,dtype=str))
    x=x[order]
    key="|".join(np.asarray(fire_ids,dtype=str)[order])
    rng=np.random.default_rng((RANDOM_SEED+zlib.crc32(key.encode()))%(2**32))
    means=np.empty(BOOTSTRAP_REPLICATES)
    for start in range(0,BOOTSTRAP_REPLICATES,128):
        stop=min(start+128,BOOTSTRAP_REPLICATES)
        means[start:stop]=x[rng.integers(0,len(x),(stop-start,len(x)))].mean(axis=1)
    return tuple(np.quantile(means,[.025,.975]))

def safe_auc(y,p,w=None):
    if len(np.unique(y))<2: return np.nan
    return float(roc_auc_score(y,p,sample_weight=w))

def fire_means(frame,values,weights):
    f=pd.DataFrame({"fire":frame["_irwin_norm"].to_numpy(),"num":np.asarray(values)*weights,"den":weights})
    g=f.groupby("fire")[["num","den"]].sum()
    return g["num"]/g["den"]

def error_summary(frame,p,baseline,model_id):
    y=frame["y"].to_numpy(dtype=float);p=np.asarray(p,dtype=float)
    b=np.broadcast_to(np.asarray(baseline,dtype=float),len(frame))
    require(len(p)==len(frame) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all(),"Invalid Held probabilities.")
    require(np.isfinite(b).all() and ((b>=0)&(b<=1)).all(),"Invalid baseline probabilities.")
    w=source_weights(frame)
    scores=fire_means(frame,(p-y)**2,w)
    base_scores=fire_means(frame,(b-y)**2,w)
    lo,hi=bootstrap_interval(scores.to_numpy(),scores.index)
    delta=scores-base_scores
    dlo,dhi=bootstrap_interval(delta.to_numpy(),delta.index)
    eps=1e-7
    pc=np.clip(p,eps,1-eps)
    losses=fire_means(frame,-(y*np.log(pc)+(1-y)*np.log(1-pc)),w)
    unit_errors=fire_means(frame,(p-y)**2,np.ones(len(frame)))
    aucs=[]
    for _,idx in frame.groupby("_irwin_norm",sort=True).indices.items():
        value=safe_auc(y[idx],p[idx],w[idx])
        if np.isfinite(value): aucs.append(value)
    row={"model_id":model_id,"n_units":len(frame),"n_source_records":frame["source_record_key"].nunique(),
         "n_fires":frame["_irwin_norm"].nunique(),"n_held":int(y.sum()),"n_burned_over":int(len(y)-y.sum()),
         "fire_source_brier":scores.mean(),"brier_ci_lower":lo,"brier_ci_upper":hi,
         "baseline_fire_source_brier":base_scores.mean(),"brier_difference_model_minus_baseline":delta.mean(),
         "difference_ci_lower":dlo,"difference_ci_upper":dhi,
         "fire_source_log_loss":losses.mean(),"pooled_unit_brier":np.mean((p-y)**2),
         "fire_average_unit_brier":unit_errors.mean(),"pooled_unit_auc":safe_auc(y,p),
         "fire_average_source_auc":np.mean(aucs) if aucs else np.nan,"n_auc_fires":len(aucs),
         "mean_predicted_held":p.mean()}
    detail=pd.DataFrame({"fire_source_brier":scores,"baseline_fire_source_brier":base_scores,
                         "brier_difference":delta}).reset_index().rename(columns={"fire":"_irwin_norm"})
    detail["model_id"]=model_id
    return row,detail

def outcome_summary(frame,weights,intervals=True):
    if len(frame)==0:
        return {"n_units":0,"n_source_records":0,"n_fires":0,"n_held":0,"n_burned_over":0,
                "source_weighted_held":np.nan,"fire_average_source_held":np.nan,
                "held_ci_lower":np.nan,"held_ci_upper":np.nan,"degenerate_fire_interval":False,
                "meets_test_support":False}
    y=frame["y"].to_numpy(dtype=float)
    scores=fire_means(frame,y,weights)
    lo,hi=bootstrap_interval(scores.to_numpy(),scores.index) if intervals else (np.nan,np.nan)
    return {"n_units":len(frame),"n_source_records":frame["source_record_key"].nunique(),"n_fires":len(scores),
            "n_held":int(y.sum()),"n_burned_over":int(len(y)-y.sum()),
            "source_weighted_held":float(np.average(y,weights=weights)),
            "fire_average_source_held":scores.mean(),"held_ci_lower":lo,"held_ci_upper":hi,
            "degenerate_fire_interval":bool(len(scores)>1 and np.ptp(scores)==0),
            "meets_test_support":bool(len(frame)>=MIN_TEST_UNITS and frame["source_record_key"].nunique()>=MIN_TEST_PARENTS and len(scores)>=MIN_TEST_FIRES)}

def within_fire_leaf_contrast(frame,in_leaf,weights):
    supports=[]
    for mask in [in_leaf,~in_leaf]:
        counts=frame.loc[mask].groupby("_irwin_norm")["source_record_key"].nunique()
        supports.append(set(counts.index[counts.ge(MIN_CONTRAST_PARENTS_PER_FIRE)]))
    common=supports[0]&supports[1]
    mask_common=frame["_irwin_norm"].isin(common).to_numpy()
    a=(~in_leaf)&mask_common;b=in_leaf&mask_common
    sa=fire_means(frame.loc[a],frame.loc[a,"y"].to_numpy(),weights[a])
    sb=fire_means(frame.loc[b],frame.loc[b,"y"].to_numpy(),weights[b])
    pair=pd.concat([sa.rename("rest"),sb.rename("rule")],axis=1).dropna().sort_index()
    diff=pair["rule"]-pair["rest"]
    lo,hi=bootstrap_interval(diff.to_numpy(),diff.index)
    return {"contrast_fires":len(pair),"rule_minus_rest_held":diff.mean(),"contrast_ci_lower":lo,"contrast_ci_upper":hi}


## 3. Train-only preprocessing and readable trees

Numerical missing values are replaced by a **training-weighted median**, with a separate missing-value indicator. The saved rule definitions include those medians. A numerical rule involving imputed values is not proof that the corresponding physical condition was observed.

Fireline-type categories are learned from training fires only. Missing and previously unseen types are reported explicitly. Every fitted tree stores exact thresholds, preprocessing, training years, source membership hashes, and training support.

Support pruning uses training data only: both children of a retained split must contain at least the configured number of fireline units, source records, and fires. A node with insufficient child support becomes a terminal group. Probabilities are recalculated from the training outcomes routed to the final groups.

Printed conditions are rounded for readability; full-precision JSON rules and their float32 input representation determine membership. The notebook independently replays the stored paths and checks agreement with the fitted tree.


In [ ]:
def weighted_median(values,weights):
    values=np.asarray(values,dtype=float);weights=np.asarray(weights,dtype=float)
    keep=np.isfinite(values)&np.isfinite(weights)&(weights>0)
    if not keep.any(): return 0.0
    order=np.argsort(values[keep],kind="stable")
    x=values[keep][order];w=weights[keep][order]
    return float(x[np.searchsorted(np.cumsum(w),w.sum()/2,side="left")])

def fit_preprocessor(train,keys,use_type,weights):
    prep={"keys":list(keys),"use_type":bool(use_type),"medians":{},"all_missing_training":{},
          "type_categories":[],"features":[],"input_dtype":"float32"}
    for key in keys:
        values=train["x_"+key].to_numpy(dtype=float)
        prep["medians"][key]=weighted_median(values,weights)
        prep["all_missing_training"][key]=bool(~np.isfinite(values).any())
        prep["features"].append({"name":key,"kind":"numeric","key":key,
                                 "label":PREDICTORS[key][1],"unit":PREDICTORS[key][2]})
        prep["features"].append({"name":key+"__missing","kind":"missing","key":key,
                                 "label":PREDICTORS[key][1]+" missing","unit":"indicator"})
    if use_type:
        cats=sorted(set(train["type_value"]) - {"__MISSING__","__UNSEEN__"})
        prep["type_categories"]=cats+["__MISSING__","__UNSEEN__"]
        for cat in prep["type_categories"]:
            prep["features"].append({"name":"type="+cat,"kind":"category","category":cat,
                                     "label":"Fireline type = "+cat,"unit":"indicator"})
    return prep

def transform(frame,prep):
    columns=[]; missing=[]
    for key in prep["keys"]:
        values=frame["x_"+key].to_numpy(dtype=float)
        bad=~np.isfinite(values)
        columns += [np.where(bad,prep["medians"][key],values),bad.astype(float)]
        missing.append(bad)
    unseen=np.zeros(len(frame),dtype=bool)
    if prep["use_type"]:
        raw=frame["type_value"].to_numpy(dtype=str)
        unseen=~np.isin(raw,prep["type_categories"])
        values=np.where(unseen,"__UNSEEN__",raw)
        columns += [(values==cat).astype(float) for cat in prep["type_categories"]]
    X=np.column_stack(columns).astype(np.float32)
    require(np.isfinite(X).all(),"Nonfinite transformed predictors.")
    return X,{"any_numeric_missing":np.column_stack(missing).any(axis=1),
              "unseen_type":unseen}

def support_counts(frame):
    return {"n_units":len(frame),"n_source_records":frame["source_record_key"].nunique(),
            "n_fires":frame["_irwin_norm"].nunique()}

def sufficient_training_support(frame,min_units,min_parents,min_fires):
    c=support_counts(frame)
    return c["n_units"]>=min_units and c["n_source_records"]>=min_parents and c["n_fires"]>=min_fires

def extract_paths(clf):
    paths={}
    def walk(node,path):
        left=int(clf.tree_.children_left[node]);right=int(clf.tree_.children_right[node])
        if left<0:
            paths[node]=path;return
        feature=int(clf.tree_.feature[node]);threshold=float(clf.tree_.threshold[node])
        walk(left,path+[{"feature_index":feature,"operator":"<=","threshold":threshold}])
        walk(right,path+[{"feature_index":feature,"operator":">","threshold":threshold}])
    walk(0,[])
    return paths

def describe_path(path,prep):
    result=[]
    for step in path:
        meta=prep["features"][step["feature_index"]]
        goes_right=step["operator"]==">"
        if meta["kind"]=="category":
            result.append("Fireline type "+("is " if goes_right else "is not ")+meta["category"])
        elif meta["kind"]=="missing":
            result.append(meta["label"]+(" = yes" if goes_right else " = no"))
        else:
            threshold=step["threshold"]
            result.append(meta["label"]+" "+step["operator"]+" "+format(threshold,".4g")+" "+meta["unit"])
    return " AND ".join(result) if result else "All eligible fireline units (no supported split)"

def replay_paths(X,paths):
    assigned=np.full(len(X),-999,dtype=int);n_assign=np.zeros(len(X),dtype=int)
    for node,path in paths.items():
        mask=np.ones(len(X),dtype=bool)
        for step in path:
            # sklearn compares float32 inputs against double-precision thresholds.
            # Promote explicitly: NumPy weak Python scalars must not round the threshold.
            col=X[:,step["feature_index"]].astype(np.float64)
            mask &= col<=step["threshold"] if step["operator"]=="<=" else col>step["threshold"]
        assigned[mask]=int(node);n_assign[mask]+=1
    require((n_assign==1).all(),"Rules overlap or leave observations unassigned.")
    return assigned

def model_fingerprint(payload):
    return hashlib.sha256(json.dumps(safe_json(payload),sort_keys=True,separators=(",",":")).encode()).hexdigest()

def fit_rule_tree(train,spec,model_id,min_units=None,min_parents=None,min_fires=None):
    min_units=MIN_LEAF_UNITS if min_units is None else min_units
    min_parents=MIN_LEAF_PARENTS if min_parents is None else min_parents
    min_fires=MIN_LEAF_FIRES if min_fires is None else min_fires
    require(len(train)>0 and train["y"].isin([0,1]).all(),"Invalid training outcomes.")
    weights=training_weights(train,spec["weight_mode"])
    prep=fit_preprocessor(train,spec["keys"],spec["use_type"],weights)
    X,_=transform(train,prep)
    clf=DecisionTreeClassifier(max_depth=MAX_DEPTH,min_samples_leaf=min_units,
                              min_weight_fraction_leaf=MIN_LEAF_WEIGHT_FRACTION,
                              random_state=RANDOM_SEED,class_weight=None,criterion="gini")
    clf.fit(X,train["y"].to_numpy(),sample_weight=weights)
    pruned=[]
    def prune(node,idx):
        left=int(clf.tree_.children_left[node]);right=int(clf.tree_.children_right[node])
        if left<0: return
        feature=int(clf.tree_.feature[node]);threshold=clf.tree_.threshold[node]
        test=X[idx,feature].astype(np.float64)<=np.float64(threshold)
        li=idx[test];ri=idx[~test]
        if not(sufficient_training_support(train.iloc[li],min_units,min_parents,min_fires) and
               sufficient_training_support(train.iloc[ri],min_units,min_parents,min_fires)):
            pruned.append(int(node))
            clf.tree_.children_left[node]=-1;clf.tree_.children_right[node]=-1
            clf.tree_.feature[node]=-2;clf.tree_.threshold[node]=-2
        else:
            prune(left,li);prune(right,ri)
    prune(0,np.arange(len(train)))
    leaf=clf.apply(X)
    paths=extract_paths(clf)
    require(np.array_equal(leaf,replay_paths(X,paths)),"Training path replay mismatch.")
    leaf_probs={};train_support={}
    for node in paths:
        mask=leaf==node
        leaf_probs[node]=float(np.average(train.loc[mask,"y"],weights=weights[mask]))
        train_support[node]={**support_counts(train.loc[mask]),"training_held_probability":leaf_probs[node],
                             "support_pass":sufficient_training_support(train.loc[mask],min_units,min_parents,min_fires)}
    baseline=float(np.average(train["y"],weights=weights))
    payload={"model_id":model_id,"spec":spec,"preprocessor":prep,
             "training_years":sorted(train["Year"].unique().tolist()),"training_fires":sorted(train["_irwin_norm"].unique()),
             "training_unit_hash":digest(train,["unit_uid","y"],["unit_uid"]),
             "training_parent_hash":hashlib.sha256("|".join(sorted(train["source_record_key"].unique())).encode()).hexdigest(),
             "training_support":support_counts(train),"baseline_held_probability":baseline,
             "paths":paths,"leaf_probabilities":leaf_probs,"leaf_training_support":train_support,
             "pruned_nodes":pruned,"max_depth":MAX_DEPTH,
             "support_thresholds":{"units":min_units,"source_records":min_parents,"fires":min_fires},
             "tree_children_left":clf.tree_.children_left.tolist(),"tree_children_right":clf.tree_.children_right.tolist(),
             "tree_feature":clf.tree_.feature.tolist(),"tree_threshold":clf.tree_.threshold.tolist(),
             "tree_classes":clf.classes_.tolist(),"tree_value":clf.tree_.value.tolist()}
    return {"clf":clf,"prep":prep,"paths":paths,"leaf_probs":leaf_probs,"baseline":baseline,
            "payload":payload,"fingerprint":model_fingerprint(payload)}

def predict_rule_tree(bundle,frame):
    X,flags=transform(frame,bundle["prep"])
    leaf=bundle["clf"].apply(X)
    require(np.array_equal(leaf,replay_paths(X,bundle["paths"])),"Test path replay mismatch.")
    p=np.array([bundle["leaf_probs"][int(node)] for node in leaf])
    return p,leaf,flags

def assert_split(train,test,chronological=False):
    require(len(train)>0 and len(test)>0,"Empty train/test population.")
    require(not(set(train["_irwin_norm"])&set(test["_irwin_norm"])),"Fire leakage.")
    require(not(set(train["source_record_key"])&set(test["source_record_key"])),"Source-record leakage.")
    require(not(set(train["unit_uid"])&set(test["unit_uid"])),"Fireline-unit leakage.")
    if chronological:
        require(int(train["Year"].max())<int(test["Year"].min()),"Future/test-year training data.")

def rule_rows(bundle,frame,leaf,flags,stage,scope):
    weights=source_weights(frame);rows=[]
    for node,path in bundle["paths"].items():
        mask=leaf==node
        row=outcome_summary(frame.loc[mask],weights[mask],intervals=stage!="training")
        numeric_path_keys={bundle["prep"]["features"][step["feature_index"]]["key"]
                           for step in path if bundle["prep"]["features"][step["feature_index"]]["kind"]=="numeric"}
        if numeric_path_keys:
            path_missing=frame[["x_"+k for k in numeric_path_keys]].isna().any(axis=1).to_numpy()
        else: path_missing=np.zeros(len(frame),dtype=bool)
        row.update(model_id=bundle["payload"]["model_id"],rule_id=bundle["payload"]["model_id"]+"__leaf_"+str(node),
                   leaf_id=node,stage=stage,scope=scope,conditions_display=describe_path(path,bundle["prep"]),
                   training_held_probability=bundle["leaf_probs"][node],
                   training_support_pass=bundle["payload"]["leaf_training_support"][node]["support_pass"],
                   unit_coverage=mask.mean(),source_weight_coverage=float(weights[mask].sum()/weights.sum()),
                   n_rule_path_imputed=int(path_missing[mask].sum()),n_any_numeric_missing=int(flags["any_numeric_missing"][mask].sum()),
                   n_unseen_type=int(flags["unseen_type"][mask].sum()))
        if stage!="training":
            row.update(within_fire_leaf_contrast(frame,mask,weights))
            lower=row["held_ci_lower"];upper=row["held_ci_upper"]
            strong=row["meets_test_support"] and not row["degenerate_fire_interval"]
            row["descriptive_majority"]="Held" if strong and lower>.5 else "Burned Over" if strong and upper<.5 else "Uncertain / limited support"
        else:
            row.update(contrast_fires=0,rule_minus_rest_held=np.nan,contrast_ci_lower=np.nan,contrast_ci_upper=np.nan,
                       descriptive_majority="Training description only")
        rows.append(row)
    return pd.DataFrame(rows)

print("Train-only preprocessing, support pruning, and rule replay helpers ready.")


## 4. Synthetic safeguards before using the fire data

Small constructed examples check parent weighting, test-only values and categories, missing-value paths, support pruning, one-class outcomes, and fixed model fingerprints. These are software tests, not scientific results.


In [ ]:
toy=units.iloc[:12].copy().reset_index(drop=True)
toy["unit_uid"]=["toy_"+str(i) for i in range(12)]
toy["row_id"]=np.arange(12)
toy["_irwin_norm"]=["F1"]*10+["F2"]*2
toy["source_record_key"]=["P1"]*9+["P2","P3","P4"]
toy["y"]=[1]*9+[0,0,1]
toy["Year"]=2020
toy["x_humidity"]=[10.]*9+[90.,np.nan,90.]
toy["type_value"]=["A"]*10+["B"]*2
w=source_weights(toy)
require(np.isclose(np.average(toy.iloc[:10]["y"],weights=w[:10]),.5),"Fragmented parent dominates toy rate.")
wf=training_weights(toy)
require(np.allclose(pd.Series(wf).groupby(toy["_irwin_norm"]).sum(),1),"Toy fires not equal weight.")
subset=toy.drop(index=[0,1,2])
require(np.allclose(pd.Series(source_weights(subset),index=subset.index).groupby(subset["source_record_key"]).sum(),1),"Subset weights not recomputed.")
spec={"name":"synthetic","keys":["humidity"],"use_type":True,"weight_mode":"fire_source","population":"all"}
bundle=fit_rule_tree(toy,spec,"synthetic",min_units=1,min_parents=1,min_fires=1)
future=toy.iloc[:3].copy()
future["x_humidity"]=[np.nan,1e6,50];future["type_value"]="UNSEEN_TEST_TYPE"
fingerprint=bundle["fingerprint"]
p1,leaf1,flags1=predict_rule_tree(bundle,future)
future["y"]=1-future["y"]
p2,leaf2,_=predict_rule_tree(bundle,future)
require(np.array_equal(p1,p2) and np.array_equal(leaf1,leaf2),"Test labels changed predictions.")
require(model_fingerprint(bundle["payload"])==fingerprint,"Prediction changed fitted model.")
require(flags1["unseen_type"].all() and flags1["any_numeric_missing"][0],"Missing/unknown flags wrong.")
all_missing=toy.copy();all_missing["x_humidity"]=np.nan
empty_prep=fit_preprocessor(all_missing,["humidity"],False,training_weights(all_missing))
require(empty_prep["all_missing_training"]["humidity"],"All-missing training feature not flagged.")
require(np.isfinite(transform(future,empty_prep)[0]).all(),"All-missing transform failed.")
for constant in [0,1]:
    single=toy.copy();single["y"]=constant
    b=fit_rule_tree(single,spec,"single_class",min_units=1,min_parents=1,min_fires=1)
    require(np.all(predict_rule_tree(b,future)[0]==constant),"Single-class Held probability is wrong.")
pruned=fit_rule_tree(toy,spec,"support_pruning",min_units=1,min_parents=99,min_fires=99)
require(len(pruned["paths"])==1,"Unsupported branches not pruned.")
require(np.isnan(safe_auc(np.ones(3),np.array([.1,.2,.3]))),"Single-class AUC must be unavailable.")
require(np.isnan(bootstrap_interval([.2],["one_fire"])[0]),"One-fire interval must be unavailable.")
# Check exact threshold membership on float32 inputs, including adjacent representable numbers.
X,_=transform(toy,bundle["prep"])
for node in np.flatnonzero(bundle["clf"].tree_.children_left>=0):
    feature=bundle["clf"].tree_.feature[node];threshold=bundle["clf"].tree_.threshold[node]
    a=X[:3].copy()
    value=np.float32(threshold)
    a[:,feature]=[np.nextafter(value,np.float32(-np.inf)),value,np.nextafter(value,np.float32(np.inf))]
    require(np.array_equal(bundle["clf"].apply(a),replay_paths(a,bundle["paths"])),"Threshold replay precision failed.")
# Adjacent float32 values expose weak-scalar threshold rounding in NumPy 2.
adjacent=np.array([np.float32(1.0000001192092896),np.float32(1.000000238418579)],dtype=np.float32)
midpoint=(float(adjacent[0])+float(adjacent[1]))/2
boundary_paths={1:[{"feature_index":0,"operator":"<=","threshold":midpoint}],
                2:[{"feature_index":0,"operator":">","threshold":midpoint}]}
require(np.array_equal(replay_paths(adjacent.reshape(-1,1),boundary_paths),[1,2]),"Float32/double midpoint replay failed.")
print("Synthetic safeguards passed.")
del toy,future,single,subset,all_missing,bundle,pruned


## 5. Fixed comparisons for this run and evaluation roles

The primary condition tree uses nine readable environmental quantities and fireline type. Fixed comparisons ask whether performance changes when:

- using humidity alone;
- excluding fireline type;
- excluding the +18 h gust predictor (mean wind speed remains);
- adding the seven broad fuel-composition fractions;
- changing training weights to equal source records rather than equal fires and source records.

The separate width pair uses **identical width-available training and test observations**: one model excludes width, the other includes it. Width is never filled in for records without a screened positive report.

All predictor sets use the same depth/support design. They are comparisons, not candidates from which the best test result is selected. Saved LightGBM predictions are included only as a reference on matched observations; their training weights and predictor sets differ, so that comparison is not a controlled one-variable experiment.


In [ ]:
MODEL_SPECS=[
 {"name":"condition_tree","keys":ENVIRONMENT_KEYS,"use_type":True,"weight_mode":"fire_source","population":"all"},
 {"name":"humidity_only","keys":["humidity"],"use_type":False,"weight_mode":"fire_source","population":"all"},
 {"name":"environment_only","keys":ENVIRONMENT_KEYS,"use_type":False,"weight_mode":"fire_source","population":"all"},
 {"name":"without_gust","keys":[k for k in ENVIRONMENT_KEYS if k!="gust"],"use_type":True,"weight_mode":"fire_source","population":"all"},
 {"name":"with_fuel_composition","keys":ENVIRONMENT_KEYS+FUEL_KEYS,"use_type":True,"weight_mode":"fire_source","population":"all"},
 {"name":"source_weight_sensitivity","keys":ENVIRONMENT_KEYS,"use_type":True,"weight_mode":"source","population":"all"},
]
if RUN_WIDTH_ANALYSIS:
    MODEL_SPECS += [
     {"name":"width_subset_without_width","keys":ENVIRONMENT_KEYS,"use_type":True,"weight_mode":"fire_source","population":"width_available"},
     {"name":"width_subset_with_width","keys":ENVIRONMENT_KEYS+["width"],"use_type":True,"weight_mode":"fire_source","population":"width_available"},
    ]
SPLITS=[{"scheme":"five_fold","split":str(fold),"test_fold":fold} for fold in range(5)]
SPLITS += [{"scheme":"forward_year","split":str(year),"test_year":year} for year in range(2021,2025)]
FROZEN_MODELS=["condition_tree"]+(["width_subset_without_width","width_subset_with_width"] if RUN_WIDTH_ANALYSIS else [])
OOF_REFERENCE_PATH=PROJECT_ROOT/"data/processed/baseline_modeling_v1/oof_predictions/E_FULL_EXTENDED__row.parquet"
FORWARD_REFERENCE_PATH=PROJECT_ROOT/"data/processed/forward_temporal_generalization_v1/FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"

def population_frame(spec):
    return units if spec["population"]=="all" else units.loc[units["width_available"]]

def split_frames(spec,split):
    frame=population_frame(spec)
    if split["scheme"]=="five_fold":
        mask=frame["outer_fold"].eq(split["test_fold"])
        return frame.loc[~mask],frame.loc[mask]
    year=split["test_year"]
    return frame.loc[frame["Year"].lt(year)],frame.loc[frame["Year"].eq(year)]

plan={"build_id":BUILD_ID,"input_run":INPUT_RUN_NAME,"models":MODEL_SPECS,"splits":SPLITS,
      "frozen_models":FROZEN_MODELS,"frozen_training_years":FROZEN_TRAIN_YEARS,"frozen_test_years":FROZEN_TEST_YEARS,
      "max_depth":MAX_DEPTH,"min_leaf_units":MIN_LEAF_UNITS,"min_leaf_parents":MIN_LEAF_PARENTS,
      "min_leaf_fires":MIN_LEAF_FIRES,"min_leaf_weight_fraction":MIN_LEAF_WEIGHT_FRACTION,
      "minimum_test_units":MIN_TEST_UNITS,"minimum_test_parents":MIN_TEST_PARENTS,"minimum_test_fires":MIN_TEST_FIRES,
      "minimum_contrast_parents_per_fire":MIN_CONTRAST_PARENTS_PER_FIRE,"width_review_ft":WIDTH_REVIEW_FT,
      "bootstrap_replicates":BOOTSTRAP_REPLICATES,"seed":RANDOM_SEED,"predictors":PREDICTORS,
      "fixed_classification_cutoff":0.5,"is_smoke_test":IS_SMOKE_TEST,
      "primary_error":"Mean across fires of source-balanced mean squared probability error",
      "prior_exploration":"All years were previously examined; held out from this fit is not independent scientific confirmation."}
write_json(RUN_DIR/"08C_fixed_run_plan.json",plan)
print("Planned refitted trees:",len(MODEL_SPECS)*len(SPLITS))
print("Additional frozen trees:",len(FROZEN_MODELS))
display(pd.DataFrame([{**s,"keys":", ".join(s["keys"])} for s in MODEL_SPECS]))


## 6. Fit and evaluate the fixed procedure

Each outer test fire is excluded from fitting, preprocessing, and support pruning. For chronological tests, all training years precede the test year.

Each checkpoint saves its exact model definition, predictions, observed leaf summaries, training support, and fit time. Leaf IDs are meaningful **only together with their model ID**. A leaf from one fold is not the same physical rule as a similarly numbered leaf from another fold.

The 0.5 classification cutoff is a fixed descriptive convention, not a tuned or recommended operational threshold.


In [ ]:
all_predictions=[]
fit_log=[];split_metrics=[];split_fire_metrics=[];all_leaf_tables=[];split_rules=[]
def prediction_frame(bundle,test,p,leaf,flags,variant,scheme,split):
    return pd.DataFrame({"row_id":test["row_id"].to_numpy(),"variant":variant,"scheme":scheme,"split":split,
                         "model_id":bundle["payload"]["model_id"],"leaf_id":leaf,
                         "p_held":p,"baseline_p":bundle["baseline"],
                         "any_numeric_missing":flags["any_numeric_missing"],"unseen_type":flags["unseen_type"]})

for spec in MODEL_SPECS:
    for split in SPLITS:
        model_id=spec["name"]+"__"+split["scheme"]+"_"+split["split"]
        train,test=split_frames(spec,split)
        assert_split(train,test,chronological=split["scheme"]=="forward_year")
        started=time.perf_counter()
        bundle=fit_rule_tree(train,spec,model_id)
        p,leaf,flags=predict_rule_tree(bundle,test)
        train_p,train_leaf,train_flags=predict_rule_tree(bundle,train)
        row,detail=error_summary(test,p,bundle["baseline"],model_id)
        row.update(variant=spec["name"],scheme=split["scheme"],split=split["split"],
                   train_years=",".join(map(str,bundle["payload"]["training_years"])),
                   baseline_p=bundle["baseline"],n_leaves=len(bundle["paths"]))
        split_metrics.append(row);split_fire_metrics.append(detail)
        leaves=pd.concat([rule_rows(bundle,train,train_leaf,train_flags,"training","training"),
                          rule_rows(bundle,test,leaf,flags,"test",split["split"])],ignore_index=True)
        leaves["variant"]=spec["name"];leaves["scheme"]=split["scheme"];leaves["split"]=split["split"]
        all_leaf_tables.append(leaves)
        pred=prediction_frame(bundle,test,p,leaf,flags,spec["name"],split["scheme"],split["split"])
        all_predictions.append(pred)
        folder=MODEL_DIR/model_id;folder.mkdir()
        write_json(folder/"model.json",{**bundle["payload"],"fingerprint":bundle["fingerprint"]})
        pred.to_parquet(folder/"test_predictions.parquet",index=False)
        leaves.to_csv(folder/"rule_outcomes.csv",index=False,encoding="utf-8-sig")
        write_json(folder/"test_metrics.json",row)
        log={"model_id":model_id,"variant":spec["name"],"scheme":split["scheme"],"split":split["split"],
             "train_units":len(train),"test_units":len(test),"train_fires":train["_irwin_norm"].nunique(),
             "test_fires":test["_irwin_norm"].nunique(),"n_leaves":len(bundle["paths"]),
             "seconds":time.perf_counter()-started,"fingerprint":bundle["fingerprint"]}
        fit_log.append(log)
        for node,path in bundle["paths"].items():
            split_rules.append({"model_id":model_id,"variant":spec["name"],"scheme":split["scheme"],
                                "split":split["split"],"leaf_id":node,"conditions_display":describe_path(path,bundle["prep"]),
                                "p_held":bundle["leaf_probs"][node]})
        print(f"{model_id}: {len(train):,} train / {len(test):,} test; "
              f"{len(bundle['paths'])} supported groups; {log['seconds']:.1f}s",flush=True)
        pd.DataFrame(fit_log).to_csv(RUN_DIR/"08C_fit_progress.csv",index=False)
predictions=pd.concat(all_predictions,ignore_index=True)
tables["split_performance"]=pd.DataFrame(split_metrics)
tables["split_fire_errors"]=pd.concat(split_fire_metrics,ignore_index=True)
tables["refitted_rule_outcomes"]=pd.concat(all_leaf_tables,ignore_index=True)
tables["refitted_rule_paths"]=pd.DataFrame(split_rules)
tables["fit_log"]=pd.DataFrame(fit_log)
require(not predictions.duplicated(["variant","scheme","row_id"]).any(),"Duplicate out-of-sample predictions.")
for spec in MODEL_SPECS:
    population=population_frame(spec)
    for scheme in ["five_fold","forward_year"]:
        expected=population if scheme=="five_fold" else population.loc[population["Year"].ge(2021)]
        got=predictions.loc[predictions["variant"].eq(spec["name"]) & predictions["scheme"].eq(scheme),"row_id"]
        require(set(got)==set(expected["row_id"]),"Incomplete test predictions for "+spec["name"]+" / "+scheme)
print("All planned out-of-sample predictions present exactly once.")


## 7. Compare errors on identical test observations

Negative model-minus-baseline Brier differences mean lower probability error than the training-only constant prediction.

For matched model comparisons, negative candidate-minus-reference differences favor the candidate. Width comparisons always use the width-covered population on both sides. Overall forward-year summaries give each fire equal weight, **not each year**.

A saved LightGBM reference is evaluated on the same population for context. It is not used to train these rules. The new error weighting differs from the paper's original summaries, so the numerical values need not match the manuscript tables.


In [ ]:
lookup=units.set_index("row_id",drop=False)
def frame_for_prediction(pred):
    frame=lookup.loc[pred["row_id"].to_numpy()].copy()
    require(np.array_equal(frame["row_id"].to_numpy(),pred["row_id"].to_numpy()),"Prediction alignment failed.")
    return frame

aggregate_rows=[];aggregate_fire_rows=[];classification_rows=[]
for (variant,scheme),pred in predictions.groupby(["variant","scheme"],sort=True):
    frame=frame_for_prediction(pred)
    row,detail=error_summary(frame,pred["p_held"].to_numpy(),pred["baseline_p"].to_numpy(),variant+"__"+scheme+"__combined")
    row.update(variant=variant,scheme=scheme,split="combined",n_leaves=np.nan)
    aggregate_rows.append(row)
    detail["variant"]=variant;detail["scheme"]=scheme
    aggregate_fire_rows.append(detail)
    y=frame["y"].to_numpy();decision=pred["p_held"].to_numpy()>=.5
    w=source_weights(frame)
    tp=w[(y==1)&decision].sum();fp=w[(y==0)&decision].sum();fn=w[(y==1)&~decision].sum()
    bas=[]
    for _,idx in frame.groupby("_irwin_norm",sort=True).indices.items():
        yy=y[idx];dd=decision[idx];ww=w[idx]
        if len(np.unique(yy))==2:
            recall=np.average(dd[yy==1],weights=ww[yy==1])
            specificity=np.average(~dd[yy==0],weights=ww[yy==0])
            bas.append((recall+specificity)/2)
    classification_rows.append({"variant":variant,"scheme":scheme,"cutoff":.5,
         "pooled_source_weighted_held_precision":tp/(tp+fp) if tp+fp else np.nan,
         "pooled_source_weighted_held_recall":tp/(tp+fn) if tp+fn else np.nan,
         "fire_average_source_balanced_accuracy":np.mean(bas) if bas else np.nan,
         "n_both_outcome_fires":len(bas),"n_predicted_held_units":int(decision.sum()),
         "n_predicted_burned_over_units":int((~decision).sum())})
tables["combined_performance"]=pd.DataFrame(aggregate_rows)
aggregate_fire=pd.concat(aggregate_fire_rows,ignore_index=True)
tables["combined_fire_errors"]=aggregate_fire
tables["classification_at_fixed_half"]=pd.DataFrame(classification_rows)

PAIRS=[
 ("condition_tree","humidity_only","Condition tree minus humidity-only"),
 ("condition_tree","environment_only","Adding fireline type"),
 ("condition_tree","without_gust","Including +18 h gust"),
 ("with_fuel_composition","condition_tree","Adding fuel composition"),
 ("source_weight_sensitivity","condition_tree","Changing training weights"),
]
if RUN_WIDTH_ANALYSIS:
    PAIRS.append(("width_subset_with_width","width_subset_without_width","Adding reported width on matched records"))
paired_rows=[]
for candidate,reference,label in PAIRS:
    for scheme in ["five_fold","forward_year"]:
        a=predictions.loc[predictions["variant"].eq(candidate)&predictions["scheme"].eq(scheme)].set_index("row_id")
        b=predictions.loc[predictions["variant"].eq(reference)&predictions["scheme"].eq(scheme)].set_index("row_id")
        require(set(a.index)==set(b.index),"Model comparison uses different test observations.")
        b=b.reindex(a.index)
        frame=lookup.loc[a.index]
        weights=source_weights(frame);y=frame["y"].to_numpy()
        fire_a=fire_means(frame,(a["p_held"].to_numpy()-y)**2,weights)
        fire_b=fire_means(frame,(b["p_held"].to_numpy()-y)**2,weights)
        for scope,ids in [("combined",fire_a.index)]+[
            (str(year),frame.loc[frame["Year"].eq(year),"_irwin_norm"].unique()) for year in sorted(frame["Year"].unique())]:
            fa=fire_a.reindex(ids);fb=fire_b.reindex(ids);delta=fa-fb
            lo,hi=bootstrap_interval(delta.to_numpy(),delta.index)
            paired_rows.append({"comparison":label,"candidate":candidate,"reference":reference,"scheme":scheme,"scope":scope,
                                "n_fires":len(delta),"candidate_brier":fa.mean(),"reference_brier":fb.mean(),
                                "difference_candidate_minus_reference":delta.mean(),"ci_lower":lo,"ci_upper":hi})
tables["paired_model_comparisons"]=pd.DataFrame(paired_rows)

reference_rows=[];reference_status=[]
for scheme,path,column in [("five_fold",OOF_REFERENCE_PATH,"pred_prob"),
                           ("forward_year",FORWARD_REFERENCE_PATH,"pred_forward_fixed_05c")]:
    if not path.is_file():
        reference_status.append({"scheme":scheme,"status":"Unavailable; no replacement fitted","path":str(path)})
        continue
    schema=set(pq.ParquetFile(path).schema_arrow.names)
    cols=[c for c in ["unit_uid","hold_label",column] if c in schema]
    require(set(["unit_uid",column]).issubset(schema),"Saved reference schema changed.")
    ref=pd.read_parquet(path,columns=cols)
    require(ref["unit_uid"].is_unique,"Duplicate saved reference IDs.")
    for population,variant in [("all","condition_tree")]+([("width_available","width_subset_without_width")] if RUN_WIDTH_ANALYSIS else []):
        pred=predictions.loc[predictions["variant"].eq(variant)&predictions["scheme"].eq(scheme)]
        frame=frame_for_prediction(pred)
        aligned=frame[["unit_uid","y"]].merge(ref,on="unit_uid",how="left",validate="one_to_one")
        require(aligned[column].notna().all(),"Missing reference probabilities.")
        if "hold_label" in aligned:
            require(aligned["y"].eq(aligned["hold_label"]).all(),"Reference labels disagree.")
        row,_=error_summary(frame,aligned[column].to_numpy(),pred["baseline_p"].to_numpy(),"saved_lightgbm_"+scheme+"_"+population)
        row.update(scheme=scheme,population=population,
                   baseline_note="Same constant as the primary 08C condition-tree comparison, not original LightGBM training prevalence")
        reference_rows.append(row)
    reference_status.append({"scheme":scheme,"status":"Aligned to evaluated observations","path":str(path)})
    input_stats[str(path)]={"bytes":path.stat().st_size,"modified_ns":path.stat().st_mtime_ns}
tables["saved_lightgbm_reference"]=pd.DataFrame(reference_rows)
tables["saved_reference_status"]=pd.DataFrame(reference_status)
display(tables["combined_performance"][["variant","scheme","fire_source_brier","baseline_fire_source_brier",
                                       "brier_difference_model_minus_baseline","difference_ci_lower","difference_ci_upper"]])
display(tables["paired_model_comparisons"].query("scope == 'combined'"))


## 8. Freeze one readable set of rules, then examine later years

For each frozen specification, train once on **2020–2022** and keep the preprocessing, conditions, probabilities, and baseline unchanged for **both 2023 and 2024**. The 2024 outcomes cannot update this tree.

Rule cards report every terminal group, including groups with few or no test observations. The label *descriptive majority* requires adequate test support and a nondegenerate interval entirely above or below 50%. It summarizes this historical sample; it is not a validated operating rule or a guarantee.

A within-fire rule-versus-rest comparison uses only fires containing at least five source records on each side. It helps distinguish a rule's association within fires from differences in which fires enter its group. It is still observational.

The primary frozen tree is also checked on narrower **test-only diagnostic subsets**: longer fireline units and source records with one prediction time and a high retained-length ratio. The rules are not refitted or chosen using these subsets.


In [ ]:
frozen_bundles={};frozen_predictions=[];frozen_metrics=[];frozen_rule_tables=[];frozen_fingerprints=[]
for name in FROZEN_MODELS:
    spec=next(s for s in MODEL_SPECS if s["name"]==name)
    frame=population_frame(spec)
    train=frame.loc[frame["Year"].isin(FROZEN_TRAIN_YEARS)]
    test=frame.loc[frame["Year"].isin(FROZEN_TEST_YEARS)]
    assert_split(train,test,chronological=True)
    model_id=name+"__frozen_2020_2022"
    bundle=fit_rule_tree(train,spec,model_id)
    frozen_bundles[name]=bundle
    folder=MODEL_DIR/model_id;folder.mkdir()
    before=bundle["fingerprint"]
    train_p,train_leaf,train_flags=predict_rule_tree(bundle,train)
    training_table=rule_rows(bundle,train,train_leaf,train_flags,"training","2020–2022")
    training_table["variant"]=name
    frozen_rule_tables.append(training_table)
    p,leaf,flags=predict_rule_tree(bundle,test)
    pred=prediction_frame(bundle,test,p,leaf,flags,name,"frozen","2023–2024")
    frozen_predictions.append(pred)
    pred.to_parquet(folder/"test_predictions.parquet",index=False)
    scopes=[("2023–2024",np.ones(len(test),dtype=bool))]
    scopes += [(str(year),test["Year"].eq(year).to_numpy()) for year in FROZEN_TEST_YEARS]
    if name=="condition_tree":
        scopes += [
         ("2023–2024: minimum 25 m",test["actual_line_length_m"].ge(25).to_numpy()),
         ("2023–2024: minimum 100 m",test["actual_line_length_m"].ge(100).to_numpy()),
         ("2023–2024: one time, high retained-length ratio",
          (test["n_prediction_times"].eq(1)&test["retained_length_ratio"].between(.8,1.02)).to_numpy()),
        ]
    for scope,mask in scopes:
        if not mask.any(): continue
        part=test.loc[mask]
        pp,ll,ff=predict_rule_tree(bundle,part)
        require(np.array_equal(pp,p[mask]) and np.array_equal(ll,leaf[mask]),"Frozen predictions changed by test subset.")
        row,_=error_summary(part,pp,bundle["baseline"],model_id+"__"+scope)
        row.update(variant=name,scope=scope,baseline_p=bundle["baseline"],fingerprint=before)
        frozen_metrics.append(row)
        rt=rule_rows(bundle,part,ll,ff,"test",scope);rt["variant"]=name
        frozen_rule_tables.append(rt)
        after=model_fingerprint(bundle["payload"])
        require(after==before,"Frozen tree changed between test years/scopes.")
        frozen_fingerprints.append({"variant":name,"scope":scope,"fingerprint":after,"unchanged":True})
    write_json(folder/"model.json",{**bundle["payload"],"fingerprint":before})
    print(model_id,":",len(bundle["paths"]),"unchanged rules tested in",FROZEN_TEST_YEARS,flush=True)

frozen_predictions=pd.concat(frozen_predictions,ignore_index=True)
tables["frozen_rule_cards"]=pd.concat(frozen_rule_tables,ignore_index=True)
tables["frozen_performance"]=pd.DataFrame(frozen_metrics)
tables["frozen_fingerprint_checks"]=pd.DataFrame(frozen_fingerprints)
primary_cards=tables["frozen_rule_cards"].loc[
    tables["frozen_rule_cards"]["variant"].eq("condition_tree")&
    tables["frozen_rule_cards"]["stage"].eq("test")&
    tables["frozen_rule_cards"]["scope"].isin(["2023","2024","2023–2024"])]
display(primary_cards[["leaf_id","scope","conditions_display","training_held_probability","fire_average_source_held",
                       "held_ci_lower","held_ci_upper","n_fires","n_source_records","n_units",
                       "n_rule_path_imputed","descriptive_majority"]])

# Simple independent check that the matched width tests use identical IDs in every scope.
if RUN_WIDTH_ANALYSIS:
    a=frozen_predictions.loc[frozen_predictions["variant"].eq("width_subset_without_width"),"row_id"]
    b=frozen_predictions.loc[frozen_predictions["variant"].eq("width_subset_with_width"),"row_id"]
    require(set(a)==set(b),"Frozen width pair has different test populations.")


## 9. Figures and plain-language rule cards

Figures show the primary frozen tree, observed Held fractions for its unchanged groups in each test year, and paired predictive-error comparisons. Full tables accompany every figure.

The tree diagram describes training. The rule-outcome chart describes held-out outcomes. These are deliberately separate. Numerical labels in the diagram are rounded; use the model JSON for exact membership.


In [ ]:
import textwrap
figure_files=[]
def save_figure(fig,name):
    if IS_SMOKE_TEST:
        fig.text(.99,.005,f"SMOKE TEST: {BOOTSTRAP_REPLICATES} resamples",ha="right",color="#B04A00",fontsize=9)
    for extension in ["png","pdf"]:
        path=FIG_DIR/(name+"."+extension)
        fig.savefig(path,bbox_inches="tight")
        figure_files.append(str(path.relative_to(RUN_DIR)))
    plt.show();plt.close(fig)

bundle=frozen_bundles["condition_tree"]
feature_names=[textwrap.fill(m["label"]+" ["+m["unit"]+"]",30) for m in bundle["prep"]["features"]]
fig,ax=plt.subplots(figsize=(20,11))
plot_tree(bundle["clf"],feature_names=feature_names,class_names=[("Burned Over" if c==0 else "Held") for c in bundle["clf"].classes_],
          filled=True,rounded=True,impurity=False,proportion=True,node_ids=True,precision=2,fontsize=8,ax=ax)
ax.set_title("Condition tree trained on 2020–2022 only\nNode values [Burned Over, Held] use equal-fire/source training weights; sample percentages count fireline units",fontsize=14)
save_figure(fig,"08C_frozen_training_tree")

cards=tables["frozen_rule_cards"]
cards=cards.loc[cards["variant"].eq("condition_tree")&cards["stage"].eq("test")]
leaf_ids=list(bundle["paths"])
fig,axes=plt.subplots(1,2,figsize=(13,max(4,0.65*len(leaf_ids))),sharex=True,sharey=True)
for ax,year,color in zip(axes,["2023","2024"],["#0072B2","#D55E00"]):
    part=cards.loc[cards["scope"].eq(year)].set_index("leaf_id").reindex(leaf_ids)
    for pos,(_,row) in enumerate(part.iterrows()):
        if np.isfinite(row["fire_average_source_held"]):
            ax.plot(row["fire_average_source_held"],pos,marker="o" if row["meets_test_support"] else "x",color=color)
            if np.isfinite(row["held_ci_lower"]):
                ax.hlines(pos,row["held_ci_lower"],row["held_ci_upper"],color=color)
        ax.text(1.025,pos,f"{int(row['n_fires'])} fires / {int(row['n_units']):,} units",va="center",fontsize=9)
    ax.axvline(.5,color="0.5",linestyle="--",linewidth=1)
    ax.set_xlim(0,1);ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_title(year);ax.set_xlabel("Observed Held fraction, fire-average and source-balanced")
axes[0].set_yticks(np.arange(len(leaf_ids)),["Rule "+str(n) for n in leaf_ids])
axes[0].invert_yaxis()
fig.suptitle("The same frozen conditions tested in two subsequent years\nBars: whole-fire 95% intervals; ×: below the configured support screen",fontsize=12)
fig.tight_layout()
save_figure(fig,"08C_frozen_rule_test_outcomes")

comparison=tables["paired_model_comparisons"].query("scope == 'combined'").copy()
labels=[x[2] for x in PAIRS]
fig,axes=plt.subplots(1,2,figsize=(14,max(4,.65*len(labels))),sharey=True,sharex=True)
for ax,scheme in zip(axes,["five_fold","forward_year"]):
    part=comparison.loc[comparison["scheme"].eq(scheme)].set_index("comparison").reindex(labels)
    ax.hlines(np.arange(len(labels)),part["ci_lower"],part["ci_upper"],color="#0072B2")
    ax.plot(part["difference_candidate_minus_reference"],np.arange(len(labels)),"o",color="#0072B2")
    ax.axvline(0,color="0.4",linestyle="--",linewidth=1)
    ax.set_title("Five-fold excluded fires" if scheme=="five_fold" else "Forward-year tests")
    ax.set_xlabel("Brier difference: candidate minus reference\nNegative values favor the candidate")
axes[0].set_yticks(np.arange(len(labels)),[textwrap.fill(x,30) for x in labels])
axes[0].invert_yaxis()
fig.suptitle("Paired comparisons use identical test observations and complete-fire resamples",fontsize=12)
fig.tight_layout()
save_figure(fig,"08C_paired_model_error_comparisons")

def render_cards(frame):
    lines=["# 08C frozen condition rules","",
           "**Exploratory historical associations, not operating instructions.**","",
           "One tree fitted on 2020–2022. Conditions and fitted probabilities are unchanged in 2023 and 2024.",
           "A source record is a mapped record, not necessarily a complete constructed fireline.",
           "Each observed Held fraction first weights source records within a fire, then gives each represented fire equal weight.",
           "The fitted probability uses training weights allocated before the tree split; the descriptive leaf rate reweights represented fires equally within the leaf. Their difference is not a calibration test.",
           "Numerical conditions below are rounded; exact thresholds and imputation values are in the model JSON.",""]
    for node,path in bundle["paths"].items():
        train_row=tables["frozen_rule_cards"].loc[
            tables["frozen_rule_cards"]["variant"].eq("condition_tree") &
            tables["frozen_rule_cards"]["stage"].eq("training") &
            tables["frozen_rule_cards"]["leaf_id"].eq(node)].iloc[0]
        lines += [f"## Rule {node}",describe_path(path,bundle["prep"]),"",
                  f"Training fitted Held probability: {bundle['leaf_probs'][node]:.1%}.",
                  f"Training descriptive Held fraction, averaged equally across fires within this group: {train_row['fire_average_source_held']:.1%}.",
                  "| Test scope | Observed Held | 95% interval | Fires | Source records | Fireline units | Missing values on rule path |",
                  "|---|---:|---:|---:|---:|---:|---:|"]
        for scope in ["2023","2024","2023–2024"]:
            row=frame.loc[(frame["leaf_id"]==node)&frame["scope"].eq(scope)].iloc[0]
            rate=f"{row['fire_average_source_held']:.1%}" if np.isfinite(row["fire_average_source_held"]) else "Unavailable"
            interval=f"{row['held_ci_lower']:.1%}–{row['held_ci_upper']:.1%}" if np.isfinite(row["held_ci_lower"]) else "Unavailable"
            lines.append(f"| {scope} | {rate} | {interval} | {row['n_fires']} | {row['n_source_records']} | {row['n_units']} | {row['n_rule_path_imputed']} |")
        lines.append("")
        for scope in ["2023","2024","2023–2024"]:
            r=frame.loc[(frame["leaf_id"]==node)&frame["scope"].eq(scope)].iloc[0]
            status="met" if r["meets_test_support"] else "NOT met"
            lines.append(f"- {scope}: {r['n_held']:,} Held / {r['n_burned_over']:,} Burned Over fireline units; support screen {status}; "
                         f"{r['n_unseen_type']:,} unseen-type observations; degenerate interval: {r['degenerate_fire_interval']}.")
        used={bundle["prep"]["features"][step["feature_index"]]["key"]
              for step in path if bundle["prep"]["features"][step["feature_index"]]["kind"]=="numeric"}
        if used:
            medians="; ".join(k+"="+format(bundle["prep"]["medians"][k],".6g") for k in sorted(used))
            lines += ["","Missing numerical values follow the rule after replacement by training medians: "+medians+"."]
        lines += ["","Sparse or degenerate groups do not establish a dependable majority outcome.",""]
    return "\n".join(lines)+"\n"
(RUN_DIR/"08C_FROZEN_RULE_CARDS.md").write_text(render_cards(cards),encoding="utf-8")
print("Saved figures and plain-language cards.")


## 10. Save the review package and verify preservation

The summary distinguishes results of the fitted procedure from the individual frozen rules. No rule is promoted automatically because it has a favorable interval. No existing data, notebook, model, or manuscript is overwritten.

Review the full rule table, including sparse groups, before deciding what belongs in the paper. Further tuning after seeing these tests would be another exploratory analysis, not independent confirmation.


In [ ]:
require(len(units)==304143 and units["unit_uid"].is_unique,"Frozen unit population changed.")
require(digest(units,original_columns,["unit_uid"])==input_hash,"Original unit values changed.")
for path,stats in input_stats.items():
    p=Path(path)
    require(p.stat().st_size==stats["bytes"] and p.stat().st_mtime_ns==stats["modified_ns"],"Input changed during run: "+path)
require(tables["frozen_fingerprint_checks"]["unchanged"].all(),"Frozen model changed.")
for name,table in tables.items():
    table.to_csv(RUN_DIR/(name+".csv"),index=False,encoding="utf-8-sig")
predictions.to_parquet(RUN_DIR/"08C_out_of_sample_predictions.parquet",index=False)
frozen_predictions.to_parquet(RUN_DIR/"08C_frozen_rule_predictions.parquet",index=False)

notebook_source_hash=saved_notebook_source_hash()
require(notebook_source_hash==NOTEBOOK_SOURCE_HASH_AT_START,"Notebook source changed during the run; restart with a saved, fixed version.")
write_json(RUN_DIR/"08C_run_manifest.json",{
    **plan,"run_id":RUN_ID,"completed_utc":datetime.now(timezone.utc).isoformat(),
    "input_files":input_stats,"input_unit_values_hash":input_hash,"notebook_source_hash":notebook_source_hash,
    "python":sys.version,"platform":platform.platform(),
    "package_versions":{package:package_version(package) for package in required},
    "n_refitted_trees":len(fit_log),"n_frozen_trees":len(frozen_bundles),
    "n_prediction_rows":len(predictions),"figures":figure_files})
write_json(RUN_DIR/"08C_quality_checks.json",{
    "is_smoke_test":IS_SMOKE_TEST,"frozen_population_preserved":True,"input_values_preserved":True,
    "original_input_stats_unchanged":True,"fire_and_parent_disjointness_checked":True,
    "chronological_training_precedes_testing":True,"preprocessing_fitted_on_training_only":True,
    "source_and_fire_weight_checks_pass":True,"test_path_replay_matches_tree":True,
    "synthetic_tests_pass":True,"out_of_sample_predictions_unique_and_complete":True,
    "frozen_models_and_baselines_unchanged":True,"width_comparisons_match_populations":True,
    "no_operational_or_causal_validation_claimed":True})

def interval_text(lo,hi,scale=1):
    return f"{scale*lo:.4f} to {scale*hi:.4f}" if np.isfinite(lo) and np.isfinite(hi) else "Unavailable"
lines=["# 08C — Interpretable condition–outcome rules","",
       "**SMOKE TEST — not a paper result.**" if IS_SMOKE_TEST else "**Exploratory historical evaluation.**","",
       f"Run {RUN_ID}; {BOOTSTRAP_REPLICATES:,} complete-fire bootstrap resamples.",
       "The new trees predict recorded Held/Burned Over outcomes directly. They are not surrogates for LightGBM.",
       "The primary fit gives equal total weight to each fire and equal weight to source records within a fire.",
       "Held is 1. No class balancing or test-selected cutoff was used.","",
       "## Primary condition-tree performance",
       "| Evaluation | Fires | Brier error | Training-only constant | Difference | 95% difference interval |",
       "|---|---:|---:|---:|---:|---:|"]
for row in tables["combined_performance"].loc[tables["combined_performance"]["variant"].eq("condition_tree")].itertuples():
    lines.append(f"| {row.scheme} | {row.n_fires} | {row.fire_source_brier:.4f} | {row.baseline_fire_source_brier:.4f} | "
                 f"{row.brier_difference_model_minus_baseline:.4f} | {interval_text(row.difference_ci_lower,row.difference_ci_upper)} |")
lines += ["","Errors average source-weighted errors within each fire and then give every fire equal weight.",
          "Negative model-minus-baseline values mean lower probability error. These weights differ from the manuscript's original unit-based summaries.","",
          "## What to inspect",
          "- frozen_rule_cards.csv and 08C_FROZEN_RULE_CARDS.md: exact same condition groups evaluated in 2023 and 2024.",
          "- frozen_performance.csv: fixed-tree transfer, including diagnostic test subsets.",
          "- paired_model_comparisons.csv: matched comparisons for type, gust, fuel composition, width, and training weights.",
          "- refitted_rule_outcomes.csv: every fold/year-specific leaf, including sparse groups.",
          "- saved_lightgbm_reference.csv: matched-population context, not a training target.",
          "- classification_at_fixed_half.csv: descriptive classifications at 0.5, not tuned operational decisions.","",
          "## Frozen rule support"]
main=tables["frozen_rule_cards"]
main=main.loc[main["variant"].eq("condition_tree")&main["stage"].eq("test")&main["scope"].eq("2023–2024")]
counts=main["descriptive_majority"].value_counts()
lines += [f"Frozen primary tree has {len(main)} groups.",
          f"Combined held-out descriptive majority counts: {counts.to_dict()}.",
          "Majority labels require the configured sample support and a nondegenerate whole-fire interval; they do not guarantee performance during a live fire.",
          "A group can have a majority Held outcome yet have more Burned Over outcomes than the other groups. Examine within-fire rule-versus-rest contrasts as well.","",
          "## Limitations and interpretation",
          "- Development and evaluation settings were fixed for this run, but all years belong to a previously explored and retrospectively selected archive.",
          "- The fixed support thresholds reduce tiny groups; they do not establish validity or eliminate multiple-comparison uncertainty.",
          "- Historical type and width availability has not been verified. Reported width is incomplete and screened, not a verified treatment measurement.",
          "- Numerical paths may contain median-imputed observations. Per-rule missing counts and training medians are saved.",
          "- Canopy and fuel fractions describe sampled pre-event landscape, not constructed treatment or measured fuel moisture.",
          "- Test-year differences can reflect environmental, fire, and mapping differences. No causal widening or construction-type effect is estimated.",
          "- Width results apply only to the width-reporting subset. Do not generalize them to missing-width records.",
          "- Source-record length, fire identifiers, year, and observation-context variables are not rule predictors.",
          "- No rule is a crew-safety or tactical suppression recommendation.","",
          "## Review before the next step",
          "Check whether the primary tree improves on its training-only baseline, whether the same frozen groups remain distinct in both years, and how much evidence supports each group.",
          "Inspect both Held and Burned Over counts; do not force a Burned Over-majority group or select only successful rules.",
          "Any further rule revision after examining these tests is exploratory. Independently collected fires are needed for confirmation.",""]
(RUN_DIR/"08C_FINDINGS_REPORT.md").write_text("\n".join(lines),encoding="utf-8")
write_json(RUN_DIR/"08C_output_inventory.json",[
    {"path":str(p.relative_to(RUN_DIR)),"bytes":p.stat().st_size}
    for p in sorted(RUN_DIR.rglob("*")) if p.is_file()])
display(Markdown("\n".join(lines)))
print("\nCompleted:",RUN_DIR)
print("For review: 08C_FINDINGS_REPORT.md, 08C_FROZEN_RULE_CARDS.md, paired_model_comparisons.csv, and frozen_rule_cards.csv.")


## How to read the run

Start with **08C_FINDINGS_REPORT.md**, then **08C_FROZEN_RULE_CARDS.md**.

A useful rule needs understandable conditions, enough supporting fires and source records, and held-out outcomes that remain informative. A high training Held probability alone is not sufficient. A large number of fireline units from a few fires is not broad validation.

The most useful paper addition may be a small set of supported condition associations, together with clear evidence of where the rules fail to transfer. The notebook does not automatically rewrite the paper or declare a deployable fireline decision rule.
